<a href="https://colab.research.google.com/github/thansihatech/CSA0816-PYTHON/blob/main/ASSIGNMENT_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import re
import sqlite3
import time
from collections import Counter

# ==========================================
# 1. SENTIMENT ANALYSIS
# ==========================================

POSITIVE = {
    "good", "clean", "fast", "helpful", "improved"
}

NEGATIVE = {
    "bad", "dirty", "slow", "rude", "smoke", "broken"
}

NEGATORS = {"not", "never", "no"}


def sentiment_score(text):

    words = re.findall(r"[a-z]+", text.lower())
    total = 0

    for i, word in enumerate(words):

        if word in POSITIVE:
            value = 1
        elif word in NEGATIVE:
            value = -1
        else:
            value = 0

        if i > 0 and words[i - 1] in NEGATORS:
            value = -value

        total += value

    return total


def sentiment_label(text):

    score = sentiment_score(text)

    if score > 0:
        return "Positive"
    elif score < 0:
        return "Negative"
    else:
        return "Neutral"


# ==========================================
# 2. TEXT SUMMARISATION
# ==========================================

STOPWORDS = {
    "the", "is", "a", "an", "and", "was",
    "were", "are", "to", "of", "in", "on",
    "from", "it", "my", "we", "our"
}


def summarise(text, k=3):

    sentences = re.split(r'(?<=[.!?])\s+', text.strip())

    words = re.findall(r"[a-z]+", text.lower())

    important = [
        word for word in words
        if word not in STOPWORDS
    ]

    frequency = Counter(important)

    ranked = []

    for i, sentence in enumerate(sentences):

        sentence_words = re.findall(
            r"[a-z]+", sentence.lower()
        )

        score = sum(
            frequency[word]
            for word in sentence_words
            if word not in STOPWORDS
        )

        ranked.append((score, i, sentence))

    selected = sorted(
        sorted(ranked, key=lambda x: (-x[0], x[1]))[:k],
        key=lambda x: x[1]
    )

    return " ".join(item[2] for item in selected)


# ==========================================
# 3. CREATE 50 TEST COMPLAINTS
# ==========================================

positive_samples = [
    "The bus service is good and fast.",
    "The station is clean and helpful.",
    "Transport service improved.",
    "The road is clean and the bus is good.",
    "Staff are helpful and service is fast."
]

negative_samples = [
    "The bus is slow and broken.",
    "The station is dirty.",
    "Smoke is bad near the station.",
    "The staff are rude.",
    "The bus is broken and slow."
]

complaints = positive_samples * 5 + negative_samples * 5

# Manually assigned expected labels
actual_labels = (
    ["Positive"] * 25 +
    ["Negative"] * 25
)


# ==========================================
# 4. TEST SENTIMENT ACCURACY
# ==========================================

correct = 0

print("\n===== SENTIMENT TEST RESULTS =====")

for i, complaint in enumerate(complaints):

    predicted = sentiment_label(complaint)
    actual = actual_labels[i]

    if predicted == actual:
        correct += 1

    print(
        i + 1,
        complaint,
        "| Predicted:", predicted,
        "| Actual:", actual
    )


accuracy = correct / len(complaints) * 100

print("\nTotal Complaints:", len(complaints))
print("Correct Predictions:", correct)
print("Agreement Percentage:", round(accuracy, 2), "%")


# ==========================================
# 5. SUMMARISE COMPLAINTS
# ==========================================

print("\n===== COMPLAINT SUMMARY =====")

all_text = " ".join(complaints)

summary = summarise(all_text, 3)

print(summary)


# ==========================================
# 6. SQLITE DATABASE CREATION
# ==========================================

print("\n===== DATABASE OPERATIONS =====")

db = sqlite3.connect(":memory:")

db.execute("""
CREATE TABLE complaints (
    id INTEGER PRIMARY KEY,
    station TEXT,
    day TEXT,
    message TEXT,
    sentiment TEXT
)
""")

# Generate 50,000 sample records

data = []

for i in range(50000):

    message = complaints[i % 50]

    station = f"S{i % 100:03}"

    day = f"2026-11-{(i % 30) + 1:02}"

    sentiment = sentiment_label(message)

    data.append((
        station,
        day,
        message,
        sentiment
    ))


# Insert records into database

db.executemany("""
INSERT INTO complaints
(station, day, message, sentiment)
VALUES (?, ?, ?, ?)
""", data)

db.commit()

print("Database created successfully")
print("Total records inserted:", len(data))


# ==========================================
# 7. SEARCH WITHOUT INDEX
# ==========================================

query = """
SELECT * FROM complaints
WHERE station = 'S050'
AND day = '2026-11-01'
"""


def measure_query():

    start = time.perf_counter()

    for _ in range(20):
        result = db.execute(query).fetchall()

    end = time.perf_counter()

    return end - start, len(result)


before_time, result_count = measure_query()

print("\nSearch without index:")
print("Matching records:", result_count)
print("Execution time:", round(before_time, 6), "seconds")


# ==========================================
# 8. CREATE DATABASE INDEX
# ==========================================

db.execute("""
CREATE INDEX idx_station_day
ON complaints(station, day)
""")

db.commit()

print("\nDatabase index created successfully")


# ==========================================
# 9. SEARCH WITH INDEX
# ==========================================

after_time, result_count = measure_query()

print("\nSearch with index:")
print("Matching records:", result_count)
print("Execution time:", round(after_time, 6), "seconds")


# ==========================================
# 10. PERFORMANCE COMPARISON
# ==========================================

print("\n===== PERFORMANCE COMPARISON =====")

print("Records:", len(data))

print("Without Index:",
      round(before_time, 6), "seconds")

print("With Index:",
      round(after_time, 6), "seconds")

if after_time < before_time:
    print("Result: Indexed search was faster.")
else:
    print("Result: Index did not improve measured time.")

# Confirm query plan
plan = db.execute(
    "EXPLAIN QUERY PLAN " + query
).fetchall()

print("\nQuery Plan:")
for row in plan:
    print(row)


# ==========================================
# 11. CUSTOM COMPLAINT INPUT
# ==========================================

print("\n===== ENTER YOUR OWN COMPLAINT =====")

user_complaint = input("Enter complaint: ")

print("Sentiment Score:",
      sentiment_score(user_complaint))

print("Sentiment:",
      sentiment_label(user_complaint))


# ==========================================
# 12. FINAL REPORT
# ==========================================

print("\n===== FINAL ENGINEERING REPORT =====")

print("Monthly complaints:", 50000)
print("Additional stations: 100")
print("Update interval: 15 minutes")

print("Sentiment agreement:",
      round(accuracy, 2), "%")

print("Summarisation: Completed")
print("Database indexing: Completed")
print("Performance testing: Completed")

print("\nB4 PROGRAM COMPLETED SUCCESSFULLY")

db.close()



===== SENTIMENT TEST RESULTS =====
1 The bus service is good and fast. | Predicted: Positive | Actual: Positive
2 The station is clean and helpful. | Predicted: Positive | Actual: Positive
3 Transport service improved. | Predicted: Positive | Actual: Positive
4 The road is clean and the bus is good. | Predicted: Positive | Actual: Positive
5 Staff are helpful and service is fast. | Predicted: Positive | Actual: Positive
6 The bus service is good and fast. | Predicted: Positive | Actual: Positive
7 The station is clean and helpful. | Predicted: Positive | Actual: Positive
8 Transport service improved. | Predicted: Positive | Actual: Positive
9 The road is clean and the bus is good. | Predicted: Positive | Actual: Positive
10 Staff are helpful and service is fast. | Predicted: Positive | Actual: Positive
11 The bus service is good and fast. | Predicted: Positive | Actual: Positive
12 The station is clean and helpful. | Predicted: Positive | Actual: Positive
13 Transport service improved